###### Gráficos Auxiliares

In [ ]:
import Pkg
Pkg.add("GraphPlot")
Pkg.add("CairoMakie")
Pkg.add("Compose")
Pkg.add("GraphMakie")
Pkg.add("Colors")
Pkg.add("CovarianceMatrices")
Pkg.add("Combinatorics")

In [1]:
using Pkg
Pkg.add("Cairo")
Pkg.add("Fontconfig")

    Updating registry at `C:\Users\User\.julia\registries\General.toml`
   Resolving package versions...
    Updating `C:\Users\User\.julia\environments\v1.11\Project.toml`
  [159f3aea] + Cairo v1.1.1
  No Changes to `C:\Users\User\.julia\environments\v1.11\Manifest.toml`
   Resolving package versions...
   Installed Fontconfig ─ v0.4.1
    Updating `C:\Users\User\.julia\environments\v1.11\Project.toml`
  [186bb1d3] + Fontconfig v0.4.1
    Updating `C:\Users\User\.julia\environments\v1.11\Manifest.toml`
  [186bb1d3] + Fontconfig v0.4.1
Precompiling project...
   1065.1 ms  ✓ Fontconfig
  1 dependency successfully precompiled in 6 seconds. 509 already precompiled.


In [3]:
URL = raw"C:\Users\User\Desktop\DAGs_CausalML\Julia\Output"


"C:\\Users\\User\\Desktop\\DAGs_CausalML\\Julia\\Output"

# Part 1a - Real life examples (2 points)


## 1.1) Explanation of Key Concepts (0.5 pts)

- **Confounder**  
  A confounder is a variable that simultaneously affects both the treatment (cause) and the outcome (effect). If not properly controlled, it generates a spurious or biased association, leading to incorrect inferences about causality.

- **Collider**  
  A collider is a variable that is influenced by two or more other variables. Conditioning on a collider (e.g., by including it as a control in regression analysis) can induce a false correlation between its determinants, thereby distorting the estimated relationships.

- **Mediator**  
  A mediator is a variable located on the causal pathway between the treatment and the outcome. It provides insight into the mechanism through which the treatment exerts its effect, helping to distinguish direct from indirect causal impacts.

## 1.2) Economics examples (1 pt)

In [11]:
using Graphs, GraphPlot, Compose
import Cairo, Fontconfig

# Grafo
g = DiGraph(3)
add_edge!(g, 1, 2)
add_edge!(g, 2, 3)
labels = ["B", "E", "Y"]

p = gplot(g,
          nodelabel = labels,
          arrowlengthfrac = 0.1,
          nodesize = 0.2,
          font_family = "sans")


# Concatenar con nombre de archivo
filename = joinpath(URL, "Confounder.png")

# Guardar
draw(PNG(filename, 6inch, 4inch), p)


### Example A: Confounder 
- **Treatment (E):** Years of education
- **Outcome (Y):** Future income
- **Confounder (B):** Family background (socioeconomic status)
- **Explanation:** Family background affects both the likelihood of obtaining more education and the level of income. If not controlled, the estimated effect of education on income will be biased.

In [15]:
using Graphs, GraphPlot, Compose
import Cairo, Fontconfig


g = DiGraph(3)
add_edge!(g, 1, 2)   # T → S
add_edge!(g, 3, 2)   # C → S

labels = ["T", "S", "C"]

x = [0.0, 1.0, 2.0]
y = [0.0, 0.0, 0.0]

gplot(g, x, y;
      nodelabel = labels,
      arrowlengthfrac = 0.1,
      nodesize = 0.2,
      font_family = "sans")


# Concatenar con nombre de archivo
filename = joinpath(URL, "Collider.png")

# Guardar
draw(PNG(filename, 6inch, 4inch), p)



### Example B: Collider  
- **Variable 1 (T):** Entrepreneurial talent  
- **Variable 2 (C):** Access to credit  
- **Collider (S):** Business survival  
- **Explanation:** Both talent and credit independently increase the chance of business survival. If we condition on survival (look only at businesses that survived), we may see a spurious negative correlation between talent and credit, even if none exists in the population.  

In [17]:
using Graphs
using GraphPlot

# Crear un grafo dirigido
g = DiGraph(3)   # 3 nodos

# Agregar aristas: B → E → Y
add_edge!(g, 1, 2)
add_edge!(g, 2, 3)

# Etiquetas de nodos
labels = ["G", "D", "Y"]

# Dibujar grafo
gplot(g,
      nodelabel = labels,
      arrowlengthfrac = 0.1,
      nodesize = 0.2,
      font_family = "sans")

# Concatenar con nombre de archivo
filename = joinpath(URL, "Mediator.png")

# Guardar
draw(PNG(filename, 6inch, 4inch), p)


### Example C: Mediator  
- **Treatment (G):** Government spending  
- **Mediator (D):** Aggregate demand  
- **Outcome (Y):** Economic growth  
- **Explanation:** Government spending boosts aggregate demand, which then fosters economic growth. Aggregate demand transmits the effect of spending onto growth, so it is a mediator.  

# Part 1b - Simpson's paradox (2 points)


In [19]:
using Random, Distributions
using DataFrames, GLM
using StatsPlots  # trae Plots y recetas útiles

# Reproducibilidad
Random.seed!(222)

# Datos simulados (dos grupos con distintos interceptos)
n = 50
XA = rand(Uniform(0, 5), n)
YA = 0.5 .* XA .+ 10 .+ rand(Normal(0, 0.5), n)

XB = rand(Uniform(5, 10), n)
YB = 0.5 .* XB .+ 2  .+ rand(Normal(0, 0.5), n)

df = DataFrame(
    X = vcat(XA, XB),
    Y = vcat(YA, YB),
    Group = vcat(fill("A", n), fill("B", n))
)

# Modelos OLS: por grupo y combinado
m_all = lm(@formula(Y ~ X), df)
m_A   = lm(@formula(Y ~ X), df[df.Group .== "A", :])
m_B   = lm(@formula(Y ~ X), df[df.Group .== "B", :])

# Predicciones sobre rango común
x_vals = collect(range(minimum(df.X), stop = maximum(df.X), length = 100))
pred_all = DataFrame(X = x_vals, Y = predict(m_all, DataFrame(X = x_vals)), Model = "Fit All")
pred_A   = DataFrame(X = x_vals, Y = predict(m_A,   DataFrame(X = x_vals)), Model = "Fit Group A")
pred_B   = DataFrame(X = x_vals, Y = predict(m_B,   DataFrame(X = x_vals)), Model = "Fit Group B")

# Gráfico
p = plot(
    title = "Simpson's Paradox Example",
    xlabel = "X", ylabel = "Y",
    legend = :topright, framestyle = :box
)

# Puntos por grupo
scatter!(df.X, df.Y, group = df.Group, ms = 4, alpha = 0.9)

# Líneas de ajuste
plot!(x_vals, pred_A.Y,   lc = :red,   ls = :dash, lw = 2, label = "Fit Group A")
plot!(x_vals, pred_B.Y,   lc = :blue,  ls = :dash, lw = 2, label = "Fit Group B")
plot!(x_vals, pred_all.Y, lc = :black,               lw = 2, label = "Fit All")

# Exportar
savefig(joinpath(URL, "simpsons_paradox.png"))



"C:\\Users\\User\\Desktop\\DAGs_CausalML\\Julia\\Output\\simpsons_paradox.png"

# Part 2 - Can we omit some controls? (7 points)


## 2.1)  Graphic DAG (1 point)

In [27]:
using Graphs, GraphPlot, Compose
import Cairo, Fontconfig   # necesarios para PNG

# Grafo
g = DiGraph(5)
add_edge!(g, 1, 2)
add_edge!(g, 1, 3)
add_edge!(g, 4, 2)
add_edge!(g, 4, 3)
add_edge!(g, 5, 4)
add_edge!(g, 5, 3)
add_edge!(g, 2, 3)

labels = ["Z1", "X", "Y", "Z2", "Z3"]

# Coordenadas
x = [0.0, 0.6, 1.0, 0.6, 0.0]
y = [0.0, 0.0, 0.0, 1.0, 1.0]

# Plot en objeto
p = gplot(g, x, y; 
          nodelabel = labels, 
          arrowlengthfrac = 0.1, 
          nodesize = 0.2)

# Guardar en archivo
filename = joinpath(URL, "OMIT.png")
draw(PNG(filename, 6inch, 4inch), p)





## 2.2) Simulation (1 point)

In [29]:
using Random, Distributions, DataFrames

# Reproducibilidad
Random.seed!(123)

# Número de observaciones
n = 5000

# Función generadora de ruido ~ N(0,1)
e() = rand(Normal(0,1), n)

# Variables
Z3 = e()
Z2 = Z3 .+ e()
Z1 = e()
X  = Z1 .+ Z2 .+ e()
Y  = X .+ Z1 .+ Z2 .+ Z3 .+ e()

# DataFrame final
df = DataFrame(Y = Y, X = X, Z1 = Z1, Z2 = Z2, Z3 = Z3)

first(df, 5)  # muestra las primeras 5 filas


Row,Y,X,Z1,Z2,Z3
,Float64,Float64,Float64,Float64,Float64
1,1.68784,0.60316,-1.08436,1.68625,0.808288
2,-0.448097,1.11268,0.346187,-0.547722,-1.12207
3,-3.70934,-1.57665,-0.15449,-1.45363,-1.10464
4,-3.0274,-1.47387,0.392698,-1.85063,-0.416993
5,1.83249,0.95159,-1.21423,1.58951,0.287588


## 2.3) Regressions (2 point)

In [31]:
using DataFrames, GLM
using CovarianceMatrices
using Distributions
using LinearAlgebra
using StatsModels
using StatsBase
using Printf

struct Fit
    model::StatsModels.TableRegressionModel
    coefs::DataFrame
end

function fit(formula, data::DataFrame)
    m = lm(formula, data)
    V = vcov(HC1(), m)
    b = coef(m)
    se = sqrt.(diag(V))
    t = b ./ se
    dof = dof_residual(m)
    p = 2 .* (1 .- cdf(TDist(dof), abs.(t)))
    dfc = DataFrame(term = coefnames(m), est = b, se = se, t = t, p = p)
    return Fit(m, dfc)
end

m1 = fit(@formula(Y ~ X), df)
m2 = fit(@formula(Y ~ X + Z1), df)
m3 = fit(@formula(Y ~ X + Z2), df)
m4 = fit(@formula(Y ~ X + Z1 + Z2), df)
m5 = fit(@formula(Y ~ X + Z1 + Z2 + Z3), df)

models = [m1, m2, m3, m4, m5]
labels = ["(1) Y~X","(2) Y~X+Z1","(3) Y~X+Z2","(4) Y~X+Z1+Z2","(5) Y~X+Z1+Z2+Z3"]

function row_from_model(f::Fit; name::AbstractString = "X")
    ct = f.coefs
    hasrow = any(ct.term .== name)
    if !hasrow
        est = se = t = p = missing
    else
        r = ct[findfirst(==(name), ct.term), :]
        est, se, t, p = r.est, r.se, r.t, r.p
    end
    z = 2.575
    ci_low  = ismissing(est) || ismissing(se) ? missing : est - z*se
    ci_high = ismissing(est) || ismissing(se) ? missing : est + z*se
    stars = ismissing(p) ? "" : p < 0.01 ? "***" : p < 0.05 ? "**" : p < 0.1 ? "*" : ""
    r2val  = r2(f.model)
    nval   = nobs(f.model)

    DataFrame(
        "Coef X" => est,
        "SE(X)"  => se,
        "CI99%"  => ismissing(ci_low) ? missing : @sprintf("[%0.3f, %0.3f]", ci_low, ci_high),
        "t(X)"   => t,
        "p(X)"   => p,
        "Sig"    => stars,
        "R2"     => r2val,
        "n"      => nval,
    )
end

summary_df = vcat([row_from_model(m) for m in models]...)
summary_df.Model = labels

# --- FIX: usar Printf.Format cuando el formato viene en variable ---
function fmt_number(x, fmt::AbstractString)
    ismissing(x) ? missing : Printf.format(Printf.Format(fmt), x)
end

fmt = transform(summary_df,
    "Coef X" => ByRow(x -> fmt_number(x, "%.3f")) => "Coef X",
    "SE(X)"  => ByRow(x -> fmt_number(x, "%.3f")) => "SE(X)",
    "t(X)"   => ByRow(x -> fmt_number(x, "%.2f")) => "t(X)",
    "p(X)"   => ByRow(x -> ismissing(x) ? missing : @sprintf("%.3g", x)) => "p(X)",
    :R2      => ByRow(x -> @sprintf("%.3f", x)) => :R2,
    :n       => ByRow(x -> string(x)) => :n
)

show(fmt, allrows=true, allcols=true)



5×9 DataFrame
 Row │ Coef X  SE(X)   CI99%           t(X)    p(X)    Sig     R2      n       Model            
     │ String  String  String          String  String  String  String  String  String           
─────┼──────────────────────────────────────────────────────────────────────────────────────────
   1 │ 2.008   0.012   [1.977, 2.039]  167.49  0       ***     0.849   5000.0  (1) Y~X
   2 │ 2.012   0.014   [1.976, 2.048]  142.99  0       ***     0.849   5000.0  (2) Y~X+Z1
   3 │ 1.523   0.014   [1.488, 1.559]  109.51  0       ***     0.899   5000.0  (3) Y~X+Z2
   4 │ 1.033   0.018   [0.988, 1.079]  58.58   0       ***     0.923   5000.0  (4) Y~X+Z1+Z2
   5 │ 1.022   0.015   [0.984, 1.059]  70.43   0       ***     0.949   5000.0  (5) Y~X+Z1+Z2+Z3

## 2.4) 99% Confidence Intervals (2 point)

In [33]:
using DataFrames, Plots
gr()  # forzar backend GR

# Helper: extraer estimador y 99% CI para "X"
function ci99(f::Fit; name::AbstractString = "X")
    ct = f.coefs
    hasrow = any(ct.term .== name)
    if !hasrow
        return DataFrame(est = missing, se = missing, lo = missing, hi = missing)
    else
        r = ct[findfirst(==(name), ct.term), :]
        est, se = r.est, r.se
        z = 2.575
        return DataFrame(est = est, se = se, lo = est - z*se, hi = est + z*se)
    end
end

labels = ["(1)", "(2)", "(3)", "(4)", "(5)"]

cis = vcat([ci99(m) for m in models]...)
cis.Model = labels

# X como posiciones numéricas
x = 1:length(labels)
y = cis.est
lo = cis.lo
hi = cis.hi

# Gráfico con puntos y barras de error
p = scatter(x, y;
    yerror = (y .- lo, hi .- y),
    markersize = 5,
    color = :blue,
    legend = false,
    xticks = (x, labels),
    xlabel = "Model specification",
    ylabel = "Coefficient of X (99% CI)",
    title = "Estimates of X with 99% Confidence Intervals",
    dpi = 300
)

# Línea de referencia en y=1
hline!([1]; linestyle = :dash, color = :red, lw = 1.5)

# Guardar
savefig(p, joinpath(URL, "DAG_Confidence_Intervals.png"))



"C:\\Users\\User\\Desktop\\DAGs_CausalML\\Julia\\Output\\DAG_Confidence_Intervals.png"

## Questions

### 2.5). Which regressions correctly estimate the effect?  (1 point)
The regressions that provide an unbiased estimate of the causal effect of $X$ on $Y$ are **(4)** and **(5)**.  
In (4) $Y \sim X + Z_{1} + Z_{2}$ all backdoors are blocked by controlling for the confounders $Z_{1}, Z_{2}$.  
In (5) $Y \sim X + Z_{1} + Z_{2} + Z_{3}$ as well, but $Z_{3}$ is redundant since it is an ancestor of $Z_{2}$.


### 2.6). Summary table (models 4 and 5) (0.5 point)

| Model | $\hat{\alpha}_{X}$ | SE($\hat{\alpha}_{X}$) | 99% CI          | $R^{2}$ | $n$   |
|:-----:|:------------------:|:----------------------:|:---------------:|:-------:|:-----:|
|  (4)  | 1.033              | 0.018                  | [0.988, 1.079]  | 0.923   | 5000  |
|  (5)  | 1.022              | 0.015                  | [0.984, 1.059]  | 0.949   | 5000  |

**Comment.** $\hat{\alpha}_{X}$ is very close to $\alpha=1$; the 99% confidence intervals include 1; the standard errors are small.  
The higher $R^{2}$ in (5) results from adding $Z_{3}$, which is unnecessary for identification.


### 2.7). Can any $Z$ be ignored?  (0.5 point)
Yes. $Z_{3}$ can be omitted. With $\{Z_{1}, Z_{2}\}$ the backdoor criterion is already satisfied; including $Z_{3}$ does not reduce bias and may harm efficiency due to collinearity.

**Conclusion.** The minimally sufficient adjustment set is $\{Z_{1}, Z_{2}\}$.



# Part 3 - Damned if you do, damned if you don't (9 points)


## 3.1) Graphic DAG (1 point) 

In [35]:
using Graphs, GraphPlot, Compose
import Cairo, Fontconfig   # necesarios para PNG

# Definir nodos y aristas
g = DiGraph(5)
labels = ["U1", "Z", "U2", "X", "Y"]

edges = [
    ("U1","Z"),
    ("U2","Z"),
    ("U1","X"),
    ("Z","Y"),
    ("U2","Y"),
    ("X","Y")
]

for (a, b) in edges
    add_edge!(g, findfirst(==(a), labels), findfirst(==(b), labels))
end

# Posiciones
x = [0.0, 1.2, 2.4, 0.0, 2.4]  
y = [1.2, 1.2, 1.2, 0.0, 0.0]

# Grafo en objeto
p = gplot(g, x, y;
          nodelabel = labels,
          arrowlengthfrac = 0.1,
          nodesize = 0.2)

# Concatenar nombre de archivo
filename = joinpath(URL, "Damned_if_you_do.png")

# Guardar
draw(PNG(filename, 6inch, 4inch), p)



## 3.2) Regressions (1 point) 

In [37]:
using Random, DataFrames
using GLM, CovarianceMatrices
using LinearAlgebra
using Distributions
using Printf
using StatsModels

# Reproducibility
Random.seed!(123)
n = 50_000

# Exogenous shocks
U1 = randn(n)
U2 = randn(n)

# DAG definition
Z = U1 .+ U2 .+ randn(n)              # collider
X = U1 .+ randn(n)                    # treatment
Y = 1 .* X .+ 1 .* U2 .+ 1 .* Z .+ randn(n)   # outcome

df = DataFrame(Y = Y, X = X, Z = Z, U1 = U1, U2 = U2)

# OLS with HC1 robust SEs
struct FitHC1
    model::StatsModels.TableRegressionModel
    coefs::DataFrame  # columns: :term, :est, :se, :t, :p
end

function fit_hc1(formula, data::DataFrame)
    m = lm(formula, data)
    V = vcov(HC1(), m)
    b = coef(m)
    se = sqrt.(diag(V))
    t = b ./ se
    dof = dof_residual(m)
    p = 2 .* (1 .- cdf(TDist(dof), abs.(t)))
    ct = DataFrame(term = coefnames(m), est = b, se = se, t = t, p = p)
    return FitHC1(m, ct)
end

# 1) Model without Z
m0 = fit_hc1(@formula(Y ~ X), df)

# 2) Model controlling for Z
m1 = fit_hc1(@formula(Y ~ X + Z), df)

# Short summary
function report(f::FitHC1, var::AbstractString = "X"; level::Float64 = 0.95)
    idx = findfirst(==(var), f.coefs.term)
    idx === nothing && return @sprintf("%s: NA (SE=NA, %d%% CI [NA, NA])", var, Int(round(level*100)))
    est = f.coefs.est[idx]
    se  = f.coefs.se[idx]
    z   = quantile(Normal(), 1 - (1 - level)/2)   # 1.96 for 95%
    lo  = est - z*se
    hi  = est + z*se
    return @sprintf("%s: %.3f (SE=%.3f, %.0f%% CI [%.3f, %.3f])",
                    var, est, se, level*100, lo, hi)
end

println("== Y ~ X ==")
println(report(m0, "X"))
println()

println("== Y ~ X + Z ==")
println(report(m1, "X"))



== Y ~ X ==
X: 1.503 (SE=0.008, 95% CI [1.487, 1.519])

== Y ~ X + Z ==
X: 0.806 (SE=0.004, 95% CI [0.798, 0.815])


## 3.3) 99% Confidence Intervals (1 point)

In [39]:
using DataFrames, Plots
gr()  # backend recomendado

# Extraer coeficientes y SE de X
ests = [m0.coefs[m0.coefs.term .== "X", :est][1],
        m1.coefs[m1.coefs.term .== "X", :est][1]]

ses  = [m0.coefs[m0.coefs.term .== "X", :se][1],
        m1.coefs[m1.coefs.term .== "X", :se][1]]

labels = ["Y ~ X", "Y ~ X + Z"]

df_plot = DataFrame(
    Model = labels,
    est   = ests,
    se    = ses
)

# Coordenadas
x = 1:length(labels)
y = df_plot.est
lo = y .- df_plot.se
hi = y .+ df_plot.se

# Graficar
p = scatter(x, y;
    yerror = (y .- lo, hi .- y),
    markersize = 5,
    color = :blue,
    legend = false,
    xticks = (x, labels),
    xlabel = "",
    ylabel = "Coefficient on X",
    title = "Estimates of βₓ with and without controlling for Z",
    dpi = 300
)

# Línea de referencia en y = 1
hline!([1.0]; linestyle = :dash, color = :red, lw = 1.5, alpha = 0.8)

# Guardar
savefig(p, joinpath(URL, "estimates_X.png"))


"C:\\Users\\User\\Desktop\\DAGs_CausalML\\Julia\\Output\\estimates_X.png"

## Opinion
The plot illustrates Simpson’s paradox in the “Damned if you do, damned if you don’t” setting.  

- When regressing $Y$ on $X$ without controls, the coefficient of $X$ is biased upward ($\hat\beta_X \approx 1.503$), far from the true causal effect of $1$.  
- When controlling for $Z$, which is a collider, the coefficient flips downward ($\hat\beta_X \approx 0.806$), also biased.  
- In both cases, the estimates fail to recover the true causal effect. This shows that adjusting for the wrong variable (a collider) can introduce bias, while omitting it does not solve the problem either.  



## 3.4) Variation of "Damned if you do, damned if you don't" - Graph & Simulation (0.5 points)

In [41]:
using Graphs, GraphPlot, Compose
import Cairo, Fontconfig   # necesarios para PNG

# Nodos y aristas
labels = ["U1","Z","U2","X","Y"]
g = DiGraph(length(labels))

edges = [
    ("U1","Z"),
    ("U2","Z"),
    ("Z","X"),
    ("U1","X"),
    ("X","Y"),
    ("Z","Y"),
    ("U2","Y"),
]

for (a,b) in edges
    add_edge!(g, findfirst(==(a), labels), findfirst(==(b), labels))
end

# Posiciones fijas
x = [0.0, 1.2, 2.4, 0.0, 2.4]
y = [1.2, 1.2, 1.2, 0.0, 0.0]

# Guardar en objeto
p = gplot(g, x, y; 
          nodelabel = labels, 
          arrowlengthfrac = 0.1, 
          nodesize = 0.2)

# Concatenar nombre de archivo
filename = joinpath(URL, "Variation_Dag.png")

# Guardar a PNG
draw(PNG(filename, 6inch, 4inch), p)


In [43]:
using Random, Distributions, DataFrames

# Reproducibilidad
Random.seed!(123)
n = 50_000

# Exogenous shocks
U1 = randn(n)
U2 = randn(n)

# Relaciones del DAG
Z = 0.8 .* U1 .+ 0.8 .* U2 .+ randn(n)          # U1→Z, U2→Z
X = 0.9 .* U1 .+ 0.7 .* Z  .+ randn(n)          # Z→X, U1→X
Y = 1.0 .* X .+ 1.0 .* Z .+ 1.0 .* U2 .+ randn(n)  # X→Y, Z→Y, U2→Y

# DataFrame final
df = DataFrame(Y = Y, X = X, Z = Z, U1 = U1, U2 = U2)

first(df, 5)  # muestra primeras 5 filas


Row,Y,X,Z,U1,U2
,Float64,Float64,Float64,Float64,Float64
1,2.05215,1.81196,1.70566,0.808288,0.725697
2,5.79131,-0.174991,1.47248,-1.12207,2.18819
3,-2.21319,-1.97019,-0.449655,-1.10464,0.559224
4,4.64066,1.74826,1.37082,-0.416993,2.4641
5,0.597484,-0.0240246,0.861442,0.287588,-1.12602


## 3.5) Regressions (1.5 points)

In [45]:
using DataFrames, GLM, CovarianceMatrices
using StatsModels, StatsBase, LinearAlgebra, Distributions, Printf
using Combinatorics   # para combinations()

# OLS con SE robustos HC1 → NamedTuple
function fit_hc1(formula, data::DataFrame)
    m = lm(formula, data)
    V = vcov(HC1(), m)
    b = coef(m)
    se = sqrt.(diag(V))
    t = b ./ se
    dof = dof_residual(m)
    p = 2 .* (1 .- cdf(TDist(dof), abs.(t)))
    ct = DataFrame(term = coefnames(m), est = b, se = se, t = t, p = p)
    return (model = m, ct = ct)
end

controls_pool = ["Z","U1","U2"]

# Todas las combinaciones de controles (incluyendo vacío)
specs = Vector{Vector{String}}([String[]])
for k in 1:length(controls_pool)
    for comb in combinations(controls_pool, k)
        push!(specs, collect(comb))
    end
end

println("== Regression results for β_X (academic format) ==")
println(repeat("-", 90))
@printf("%-20s %10s %10s %22s %10s %8s %6s\n",
        "Model", "β_X", "SE", "95% CI", "p-value", "R²", "n")
println(repeat("-", 90))

models = Dict{String, NamedTuple}()

for comb in specs
    label = isempty(comb) ? "No controls" : join(comb, " + ")
    rhs = isempty(comb) ? "X" : "X + " * join(comb, " + ")
    
    # convertir a símbolos para Term()
    rhs_terms = Term.(Symbol.(split(rhs, " + ")))
    fml = Term(:Y) ~ reduce(+, rhs_terms)

    m = fit_hc1(fml, df)
    models[label] = m

    ix = findfirst(==("X"), m.ct.term)
    est = m.ct.est[ix]
    se  = m.ct.se[ix]
    p   = m.ct.p[ix]
    z   = quantile(Normal(), 0.975)
    lo  = est - z*se
    hi  = est + z*se
    r2v = r2(m.model)
    nv  = nobs(m.model)

    @printf("%-20s %10.4f %10.4f [%6.4f, %6.4f] %10.3g %8.3f %6d\n",
            label, est, se, lo, hi, p, r2v, nv)
end


== Regression results for β_X (academic format) ==
------------------------------------------------------------------------------------------
Model                       β_X         SE                 95% CI    p-value       R²      n
------------------------------------------------------------------------------------------
No controls              1.7335     0.0044 [1.7249, 1.7421]          0    0.758  50000
Z                        0.8473     0.0046 [0.8382, 0.8564]          0    0.892  50000
U1                       1.9452     0.0063 [1.9328, 1.9577]          0    0.768  50000
U2                       1.5180     0.0031 [1.5120, 1.5240]          0    0.891  50000
Z + U1                   1.0050     0.0057 [0.9938, 1.0161]          0    0.896  50000
Z + U2                   1.0006     0.0037 [0.9934, 1.0078]          0    0.936  50000
U1 + U2                  1.4725     0.0048 [1.4631, 1.4819]          0    0.892  50000
Z + U1 + U2              1.0004     0.0045 [0.9916, 1.0091]      

## 3.6) Summary Table ( Coef and SE) (1 points)

In [59]:
using DataFrames

get_ct(m) = hasproperty(m, :ct) ? getfield(m, :ct) :
            (m isa NamedTuple && haskey(m, :ct)) ? m[:ct] :
            (m isa AbstractDict && haskey(m, :ct)) ? m[:ct] :
            error("El objeto no tiene :ct")

function stored_estimates(models::AbstractDict{<:AbstractString,<:Any})
    out = DataFrame(:Controls => String[],
                    Symbol("β_X") => Float64[],
                    Symbol("SE(β_X)") => Float64[])

    for (label, m) in models
        ct_raw = get_ct(m)
        df = ct_raw isa DataFrame ? ct_raw : DataFrame(ct_raw)

        # detectar columnas
        termcol = (:term in names(df)) ? :term :
                  (:Term in names(df)) ? :Term : names(df)[1]
        estcol  = (:estimate in names(df)) ? :estimate :
                  (:Estimate in names(df)) ? :Estimate : names(df)[2]
        secol   = (:se in names(df)) ? :se :
                  (:StdError in names(df)) ? :StdError : names(df)[3]

        ix = findfirst(==("X"), df[!, termcol])
        est = ix === nothing ? NaN : df[ix, estcol]
        se  = ix === nothing ? NaN : df[ix, secol]

        push!(out, (; Controls=label, Symbol("β_X")=>est, Symbol("SE(β_X)")=>se))
    end

    pluscount(s) = count(==('+'), s)
    cov_count = [r.Controls == "No controls" ? 0 : pluscount(r.Controls) + 1 for r in eachrow(out)]
    ord = sortperm(1:nrow(out), by=i->(out.Controls[i] != "No controls", cov_count[i], out.Controls[i]))
    out = out[ord, :]

    for c in names(out)
        if eltype(out[!, c]) <: Real
            out[!, c] = round.(out[!, c], digits=4)
        end
    end
    return out
end

results_df = stored_estimates(models)
# opción B: en Jupyter
display(results_df)


Row,Controls,β_X,SE(β_X)
,String,Float64,Float64
1,No controls,1.7335,0.0044
2,U1,1.9452,0.0063
3,U2,1.518,0.0031
4,Z,0.8473,0.0046
5,U1 + U2,1.4725,0.0048
6,Z + U1,1.005,0.0057
7,Z + U2,1.0006,0.0037
8,Z + U1 + U2,1.0004,0.0045


## 3.7) .text and .txt format ( Coef and SE) (1.5 points)

In [61]:
using DataFrames, GLM, CovarianceMatrices
using StatsModels, StatsBase, LinearAlgebra, Distributions, Printf
using Combinatorics   # combinations()
# Asume que ya tienes: df :: DataFrame con columnas Y, X, Z, U1, U2
# Asume que tienes: URL :: String con la carpeta de salida

# --- OLS con SE robustos HC1 → NamedTuple ---
function fit_hc1(formula, data::DataFrame)
    m = lm(formula, data)
    V = vcov(HC1(), m)
    b = coef(m)
    se = sqrt.(diag(V))
    t = b ./ se
    dof = dof_residual(m)
    p = 2 .* (1 .- cdf(TDist(dof), abs.(t)))
    ct = DataFrame(term = coefnames(m), est = b, se = se, t = t, p = p)
    return (model = m, ct = ct)
end

# 1) Todas las combinaciones de controles {Z,U1,U2}
controls_pool = ["Z","U1","U2"]
specs = Vector{Vector{String}}([String[]])                # "No controls"
for k in 1:length(controls_pool)
    for comb in combinations(controls_pool, k)
        push!(specs, collect(comb))
    end
end

# 2) Estimar y recolectar β y SE de X
rows = DataFrame(Controls = String[], β = Float64[], SE = Float64[])
for comb in specs
    label = isempty(comb) ? "No controls" : join(comb, ", ")
    rhs   = isempty(comb) ? "X" : "X + " * join(comb, " + ")
    fml   = Term(:Y) ~ reduce(+, Term.(Symbol.(split(rhs, " + "))))
    m     = fit_hc1(fml, df)

    ix    = findfirst(==("X"), m.ct.term)
    est   = m.ct.est[ix]
    se    = m.ct.se[ix]

    push!(rows, (; Controls = label, β = est, SE = se))
end
table = rows

# 3) Orden: "No controls" primero, luego #controles, luego alfabético
count_controls(s::AbstractString) = s == "No controls" ? 0 :
    length(split(s, r",\s*"))
ord = sortperm(1:nrow(table);
    by = i -> (table.Controls[i] != "No controls",
               count_controls(table.Controls[i]),
               table.Controls[i]))
table = table[ord, :]

# 4) Redondear a 4 decimales
table.β  = round.(table.β,  digits = 4)
table.SE = round.(table.SE, digits = 4)

# 5) Exportar TXT (tab) y LaTeX
txt_path = joinpath(URL, "table_beta_se.txt")
tex_path = joinpath(URL, "table_beta_se.tex")

# TXT con "Controls" como índice (primera columna sin encabezado adicional)
open(txt_path, "w") do io
    # encabezados con índice vacío para emular rownames
    @printf(io, "\t%s\t%s\n", "β", "SE")
    for i in 1:nrow(table)
        @printf(io, "%s\t%.4f\t%.4f\n", table.Controls[i], table.β[i], table.SE[i])
    end
end

# LaTeX tabular simple, columnas: rownames + β + SE (sin flotante)
open(tex_path, "w") do io
    println(io, "\\begin{tabular}{lrr}")
    println(io, "\\hline")
    println(io, " & \$\\beta\$ & SE \\\\")
    println(io, "\\hline")
    for i in 1:nrow(table)
        # no sanitizamos los rownames
        @printf(io, "%s & %.4f & %.4f \\\\\n", table.Controls[i], table.β[i], table.SE[i])
    end
    println(io, "\\hline")
    println(io, "\\end{tabular}")
end

# 6) Mostrar en consola
@printf("Saved TXT: %s\nSaved LaTeX: %s\n", txt_path, tex_path)
println("\n== Stored estimates (β and SE) ==")
show(table; allrows = true, allcols = true)



Saved TXT: C:\Users\User\Desktop\DAGs_CausalML\Julia\Output\table_beta_se.txt
Saved LaTeX: C:\Users\User\Desktop\DAGs_CausalML\Julia\Output\table_beta_se.tex

== Stored estimates (β and SE) ==
8×3 DataFrame
 Row │ Controls     β        SE      
     │ String       Float64  Float64 
─────┼───────────────────────────────
   1 │ No controls   1.7335   0.0044
   2 │ U1            1.9452   0.0063
   3 │ U2            1.518    0.0031
   4 │ Z             0.8473   0.0046
   5 │ U1, U2        1.4725   0.0048
   6 │ Z, U1         1.005    0.0057
   7 │ Z, U2         1.0006   0.0037
   8 │ Z, U1, U2     1.0004   0.0045

# 3.8) Based on your findings, in what way(s) can you get a good estimate of the causal effect? (0.5 point)

From the regression results, we know the true causal effect of $X$ on $Y$ is $1$.  
- Models without controls or with only one control ($Z$, $U1$, $U2$) are **biased**, as their estimates are far from $1$ (e.g., $1.7335$, $0.8473$, $1.9452$, $1.518$).  
- Good estimates (close to $1$ with narrow CIs) are obtained only when controlling simultaneously for $Z$ and at least one of the true confounders ($U1$, $U2$).  
- For instance:  
  - $Y \sim X + Z + U1 \;\;\; \hat\beta_X \approx 1.005$  
  - $Y \sim X + Z + U2 \;\;\; \hat\beta_X \approx 1.0006$  
  - $Y \sim X + Z + U1 + U2 \;\;\; \hat\beta_X \approx 1.9994$  

All these estimates are very close to the true value of $1$ and unbiased within sampling error.



# 3.9) What is the minimal sufficient set of controls to get a good estimate? (0.5 point)

The minimal sufficient adjustment set is **$\{Z, U1\}$** or alternatively **$\{Z, U2\}$**.  
Controlling only $Z$ introduces collider bias, and controlling only $U1$ or $U2$ leaves confounding unaddressed. But once $Z$ and one of the confounders are included, the backdoor paths are blocked, yielding unbiased estimates of $\beta_X$.



# 3.10) Provide intuition on why you can get good estimates controlling for the variables you established above (0.5 point)

- $U1$ and $U2$ are true confounders because they influence both $X$ and $Y$.  
- $Z$ is a collider that also has a direct effect on $Y$ and an effect on $X$. If omitted, $Z$ creates bias in the regression of $Y$ on $X$.  
- By controlling for $Z$ and one of the confounders ($U1$ or $U2$), we block the spurious paths:  
  - $X \leftarrow U1 \rightarrow Y$  
  - $X \leftarrow Z \leftarrow U2 \rightarrow Y$  
- This ensures that the only remaining path from $X$ to $Y$ is the **causal path**, so the regression recovers the true causal effect $\beta_X = 1$.  
